# FinBERT Multi-Task Fine-Tuning

Learn how to fine-tune **ProsusAI/finbert** so one model returns four outputs for financial text:

| Output | Type | Example |
|--------|------|---------|
| **sentiment** | classification | `positive` / `neutral` / `negative` |
| **sentiment_score** | regression | `-1.0` (bearish) → `+1.0` (bullish) |
| **market_impact** | classification | `low` / `medium` / `high` |
| **sector_impact** | classification | `low` / `medium` / `high` |

## What you will learn
1. Load a finance-domain BERT checkpoint (`ProsusAI/finbert`)
2. Build a **multi-task head** (shared encoder + 4 task heads)
3. Train on **Financial PhraseBank** (real sentiment labels)
4. Add **weak labels** for market/sector impact (starter approach)
5. Run inference on headlines and save the model locally

> **Note:** Financial PhraseBank only has sentiment labels. Market/sector impact uses keyword heuristics for training bootstrap. For production, replace weak labels with human-annotated data (`experiment/data/sample_news_labeled.csv` shows the schema).

## 0. Setup — fix "Pending" kernel in Cursor

If cells stay **Pending**, the notebook kernel is not connected. Do this **exact order**:

1. **Cmd+Shift+P** → **Python: Select Interpreter**
2. Choose **`./backend/.venv/bin/python`** (Python 3.12.13)
3. **Cmd+Shift+P** → **Notebook: Select Notebook Kernel**
4. Choose **Python Environments...** → same **`backend/.venv`** (NOT Homebrew python3.12)
5. **Cmd+Shift+P** → **Developer: Reload Window**
6. Run the **Kernel check** cell only

Still stuck? Use Jupyter in browser instead (always works):
```bash
cd backend && source .venv/bin/activate
jupyter notebook experiment/FinBert_Fine_tune.ipynb
```

In [10]:
# Quick kernel check — this cell should finish in <1 second
import sys
print("Kernel OK:", sys.executable)
assert ".venv" in sys.executable, (
    "Wrong kernel! Select: Python 3.12 (investment-agent .venv)"
)

Kernel OK: /Users/mandes-mega/investment_agent/backend/.venv/bin/python


In [11]:
# Install ML deps only if missing (skip if already installed)
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("transformers") is None:
    req = Path("experiment/requirements-ml.txt")
    if not req.exists():
        req = Path("requirements-ml.txt")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    print("Installed ML dependencies")
else:
    print("ML dependencies already installed — skipping pip")

ML dependencies already installed — skipping pip


In [1]:
import json
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset, concatenate_datasets, load_dataset
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

EXPERIMENT_DIR = Path(".").resolve()
if EXPERIMENT_DIR.name != "experiment":
    EXPERIMENT_DIR = Path.cwd() / "experiment"

DATA_DIR = EXPERIMENT_DIR / "data"
MODEL_DIR = EXPERIMENT_DIR / "models" / "finbert-multitask"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "ProsusAI/finbert"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")
print(f"Saving model to: {MODEL_DIR}")

Matplotlib is building the font cache; this may take a moment.
/Users/mandes-mega/investment_agent/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps
Saving model to: /Users/mandes-mega/investment_agent/backend/experiment/models/finbert-multitask


## 1. Label schema

We use consistent label maps so training and inference stay aligned.

In [12]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_TO_ID = {label: idx for idx, label in enumerate(SENTIMENT_LABELS)}
ID_TO_SENTIMENT = {idx: label for label, idx in SENTIMENT_TO_ID.items()}

IMPACT_LABELS = ["low", "medium", "high"]
IMPACT_TO_ID = {label: idx for idx, label in enumerate(IMPACT_LABELS)}
ID_TO_IMPACT = {idx: label for label, idx in IMPACT_TO_ID.items()}

# Map PhraseBank integer labels -> sentiment string
PHRASEBANK_ID_TO_SENTIMENT = {0: "negative", 1: "neutral", 2: "positive"}
SENTIMENT_TO_SCORE = {"negative": -1.0, "neutral": 0.0, "positive": 1.0}

print("Sentiment:", SENTIMENT_LABELS)
print("Impact:", IMPACT_LABELS)

Sentiment: ['negative', 'neutral', 'positive']
Impact: ['low', 'medium', 'high']


## 2. Load training data

### 2a. Financial PhraseBank (sentiment ground truth)
Standard benchmark: ~2,264 sentences where all annotators agreed on sentiment.

### 2b. Weak labels for market / sector impact
No public dataset provides market/sector impact for every sentence. We bootstrap with keyword heuristics, then optionally merge your own CSV labels.

In [13]:
MARKET_KEYWORDS = [
    "market", "fed", "federal reserve", "interest rate", "gdp", "inflation",
    "recession", "economy", "central bank", "treasury", "s&p", "nasdaq", "dow",
    "rate hike", "rate cut", "monetary policy", "systemic",
]

SECTOR_KEYWORDS = [
    "sector", "industry", "semiconductor", "chip", "banking", "bank", "healthcare",
    "pharma", "energy", "oil", "retail", "automotive", "tech", "software",
    "mining", "insurance", "reit", "biotech", "aerospace",
]


def _keyword_hits(text: str, keywords: list[str]) -> int:
    text_lower = text.lower()
    return sum(1 for kw in keywords if kw in text_lower)


def weak_label_market_impact(text: str, sentiment: str) -> str:
    hits = _keyword_hits(text, MARKET_KEYWORDS)
    if hits >= 2:
        return "high"
    if hits == 1 or sentiment != "neutral":
        return "medium"
    return "low"


def weak_label_sector_impact(text: str, sentiment: str) -> str:
    hits = _keyword_hits(text, SECTOR_KEYWORDS)
    if hits >= 2:
        return "high"
    if hits == 1:
        return "medium"
    # company-specific news with strong sentiment still moves a sector somewhat
    if sentiment != "neutral":
        return "medium"
    return "low"


def enrich_row(text: str, sentiment: str, market_impact: str | None = None, sector_impact: str | None = None) -> dict:
    market_impact = market_impact or weak_label_market_impact(text, sentiment)
    sector_impact = sector_impact or weak_label_sector_impact(text, sentiment)
    return {
        "text": text,
        "sentiment": sentiment,
        "sentiment_id": SENTIMENT_TO_ID[sentiment],
        "sentiment_score": SENTIMENT_TO_SCORE[sentiment],
        "market_impact": market_impact,
        "market_impact_id": IMPACT_TO_ID[market_impact],
        "sector_impact": sector_impact,
        "sector_impact_id": IMPACT_TO_ID[sector_impact],
    }

In [14]:
# Financial PhraseBank (100% annotator agreement subset)
# takala/financial_phrasebank uses legacy dataset scripts; this parquet version works with current `datasets`
phrasebank_ds = load_dataset(
    "gtfintechlab/financial_phrasebank_sentences_allagree", "5768"
)
phrasebank = concatenate_datasets([phrasebank_ds["train"], phrasebank_ds["test"]])

phrasebank_rows = []
for row in phrasebank:
    sentiment = PHRASEBANK_ID_TO_SENTIMENT[int(row["label"])]
    phrasebank_rows.append(enrich_row(row["sentence"], sentiment))

df_phrasebank = pd.DataFrame(phrasebank_rows)
print(f"Financial PhraseBank rows: {len(df_phrasebank)}")
df_phrasebank.head(3)

Financial PhraseBank rows: 2264


,text,sentiment,sentiment_id,sentiment_score,market_impact,market_impact_id,sector_impact,sector_impact_id
0,The equipment will be made at Vaahto 's plant ...,neutral,1,0.0,low,0,low,0
1,27 January 2011 - Finnish IT solutions provide...,positive,2,1.0,medium,1,medium,1
2,"( ADP News ) - Sep 30 , 2008 - Finnish securit...",positive,2,1.0,medium,1,medium,1


In [5]:
# Optional: merge hand-labeled examples (better market/sector supervision)
sample_csv = DATA_DIR / "sample_news_labeled.csv"
if sample_csv.exists():
    df_custom = pd.read_csv(sample_csv)
    custom_rows = [
        enrich_row(
            row["text"],
            row["sentiment"],
            market_impact=row["market_impact"],
            sector_impact=row["sector_impact"],
        )
        for _, row in df_custom.iterrows()
    ]
    df_custom = pd.DataFrame(custom_rows)
    # Override sentiment_score if provided in CSV
    if "sentiment_score" in pd.read_csv(sample_csv).columns:
        df_custom["sentiment_score"] = pd.read_csv(sample_csv)["sentiment_score"].values
    df_all = pd.concat([df_phrasebank, df_custom], ignore_index=True)
    print(f"Added {len(df_custom)} custom labeled rows")
else:
    df_all = df_phrasebank

print(f"Total training rows: {len(df_all)}")
df_all[["sentiment", "market_impact", "sector_impact"]].value_counts().head(10)

Added 6 custom labeled rows
Total training rows: 2270


sentiment  market_impact  sector_impact
neutral    low            low              1141
positive   medium         medium            567
negative   medium         medium            298
neutral    low            medium            108
           medium         low                89
           low            high               32
           medium         medium             18
positive   medium         high                5
negative   medium         high                3
neutral    medium         high                2
Name: count, dtype: int64

## 3. Multi-task FinBERT architecture

```
Input text
    ↓
FinBERT encoder (shared)
    ↓
┌──────────────┬─────────────────┬──────────────────┬──────────────────┐
│ Sentiment CE │ Score regression│ Market impact CE │ Sector impact CE │
└──────────────┴─────────────────┴──────────────────┴──────────────────┘
```

Total loss = CE(sentiment) + MSE(score) + CE(market) + CE(sector)

In [6]:
class MultiTaskFinBERT(nn.Module):
    def __init__(self, model_name: str = BASE_MODEL):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.sentiment_head = nn.Linear(hidden, len(SENTIMENT_LABELS))
        self.score_head = nn.Linear(hidden, 1)
        self.market_head = nn.Linear(hidden, len(IMPACT_LABELS))
        self.sector_head = nn.Linear(hidden, len(IMPACT_LABELS))

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.pooler_output)
        return {
            "sentiment_logits": self.sentiment_head(pooled),
            "sentiment_score": self.score_head(pooled).squeeze(-1),
            "market_logits": self.market_head(pooled),
            "sector_logits": self.sector_head(pooled),
        }


tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = MultiTaskFinBERT(BASE_MODEL).to(DEVICE)
print(model.__class__.__name__, "ready")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 86709.59it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MultiTaskFinBERT ready


In [7]:
train_df, val_df = train_test_split(
    df_all,
    test_size=0.15,
    random_state=SEED,
    stratify=df_all["sentiment_id"],
)


def tokenize_batch(texts: list[str]) -> dict:
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


class FinNewsDataset(torch.utils.data.Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx: int):
        row = self.frame.iloc[idx]
        encoded = tokenize_batch([row["text"]])
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item["sentiment_id"] = torch.tensor(row["sentiment_id"], dtype=torch.long)
        item["sentiment_score"] = torch.tensor(row["sentiment_score"], dtype=torch.float)
        item["market_impact_id"] = torch.tensor(row["market_impact_id"], dtype=torch.long)
        item["sector_impact_id"] = torch.tensor(row["sector_impact_id"], dtype=torch.long)
        return item


train_loader = DataLoader(FinNewsDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(FinNewsDataset(val_df), batch_size=BATCH_SIZE)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

Train: 1929 | Val: 341


In [8]:
ce_loss = nn.CrossEntropyLoss()
mse_loss = nn.MSELoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)


def compute_loss(batch_outputs, batch):
    loss_sentiment = ce_loss(batch_outputs["sentiment_logits"], batch["sentiment_id"])
    loss_score = mse_loss(batch_outputs["sentiment_score"], batch["sentiment_score"])
    loss_market = ce_loss(batch_outputs["market_logits"], batch["market_impact_id"])
    loss_sector = ce_loss(batch_outputs["sector_logits"], batch["sector_impact_id"])
    return loss_sentiment + loss_score + loss_market + loss_sector


@torch.no_grad()
def evaluate(loader):
    model.eval()
    preds_sentiment, labels_sentiment = [], []
    preds_market, labels_market = [], []
    preds_sector, labels_sector = [], []
    score_preds, score_labels = [], []
    total_loss = 0.0

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(batch["input_ids"], batch["attention_mask"])
        total_loss += compute_loss(outputs, batch).item()

        preds_sentiment.extend(outputs["sentiment_logits"].argmax(-1).cpu().tolist())
        labels_sentiment.extend(batch["sentiment_id"].cpu().tolist())
        preds_market.extend(outputs["market_logits"].argmax(-1).cpu().tolist())
        labels_market.extend(batch["market_impact_id"].cpu().tolist())
        preds_sector.extend(outputs["sector_logits"].argmax(-1).cpu().tolist())
        labels_sector.extend(batch["sector_impact_id"].cpu().tolist())
        score_preds.extend(outputs["sentiment_score"].cpu().tolist())
        score_labels.extend(batch["sentiment_score"].cpu().tolist())

    return {
        "loss": total_loss / len(loader),
        "sentiment_acc": accuracy_score(labels_sentiment, preds_sentiment),
        "sentiment_f1": f1_score(labels_sentiment, preds_sentiment, average="macro"),
        "market_acc": accuracy_score(labels_market, preds_market),
        "sector_acc": accuracy_score(labels_sector, preds_sector),
        "score_mae": mean_absolute_error(score_labels, score_preds),
    }

In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(batch["input_ids"], batch["attention_mask"])
        loss = compute_loss(outputs, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    metrics = evaluate(val_loader)
    metrics["epoch"] = epoch
    metrics["train_loss"] = running_loss / len(train_loader)
    history.append(metrics)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={metrics['train_loss']:.3f} val_loss={metrics['loss']:.3f} | "
        f"sentiment_acc={metrics['sentiment_acc']:.3f} score_mae={metrics['score_mae']:.3f} | "
        f"market_acc={metrics['market_acc']:.3f} sector_acc={metrics['sector_acc']:.3f}"
    )

Epoch 1/3 | train_loss=0.663 val_loss=0.607 | sentiment_acc=0.971 score_mae=0.090 | market_acc=0.956 sector_acc=0.918


In [ ]:
hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_df["epoch"], hist_df["sentiment_acc"], marker="o", label="sentiment")
axes[0].plot(hist_df["epoch"], hist_df["market_acc"], marker="o", label="market impact")
axes[0].plot(hist_df["epoch"], hist_df["sector_acc"], marker="o", label="sector impact")
axes[0].set_title("Validation accuracy")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["score_mae"], marker="o", color="tab:red")
axes[1].set_title("Sentiment score MAE (lower is better)")
axes[1].set_xlabel("epoch")
plt.tight_layout()
plt.show()

## 4. Inference helper

This is the function you would call from your investment agent after loading saved weights.

In [ ]:
@torch.no_grad()
def predict_financial_text(text: str, model=model, tokenizer=tokenizer) -> dict:
    model.eval()
    encoded = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
    outputs = model(encoded["input_ids"], encoded["attention_mask"])

    sentiment_probs = torch.softmax(outputs["sentiment_logits"], dim=-1).squeeze(0)
    market_probs = torch.softmax(outputs["market_logits"], dim=-1).squeeze(0)
    sector_probs = torch.softmax(outputs["sector_logits"], dim=-1).squeeze(0)

    sentiment_id = int(sentiment_probs.argmax())
    market_id = int(market_probs.argmax())
    sector_id = int(sector_probs.argmax())

    # Blend classifier confidence with regression head for a smoother score
    reg_score = float(outputs["sentiment_score"].squeeze().cpu())
    cls_score = float(sentiment_probs[2] - sentiment_probs[0])  # P(pos) - P(neg)
    blended_score = round(0.5 * reg_score + 0.5 * cls_score, 4)

    return {
        "text": text,
        "sentiment": ID_TO_SENTIMENT[sentiment_id],
        "sentiment_confidence": round(float(sentiment_probs[sentiment_id]), 4),
        "sentiment_score": blended_score,
        "market_impact": ID_TO_IMPACT[market_id],
        "market_impact_confidence": round(float(market_probs[market_id]), 4),
        "sector_impact": ID_TO_IMPACT[sector_id],
        "sector_impact_confidence": round(float(sector_probs[sector_id]), 4),
    }

In [ ]:
demo_headlines = [
    "The Federal Reserve signaled another rate hike amid sticky inflation.",
    "NVIDIA raised guidance as AI chip demand accelerated across cloud providers.",
    "Company confirmed it will repurchase shares; no change to dividend policy.",
]

for headline in demo_headlines:
    result = predict_financial_text(headline)
    print(json.dumps(result, indent=2))
    print("-" * 60)

## 5. Try on live Yahoo Finance news (optional)

Uses the same news source as your agent's `fetch_fundamental_data_and_news` tool.

In [ ]:
import yfinance as yf

TICKER = "AAPL"  # change to any symbol

news_items = yf.Ticker(TICKER).news or []
print(f"Found {len(news_items)} news items for {TICKER}\n")

for item in news_items[:3]:
    content = item.get("content", item)
    title = content.get("title") or content.get("headline") or ""
    summary = content.get("summary") or ""
    text = f"{title}. {summary}".strip()
    if not text:
        continue
    print(predict_financial_text(text))
    print()

## 6. Save & reload model

In [ ]:
torch.save(model.state_dict(), MODEL_DIR / "multitask_head.pt")
tokenizer.save_pretrained(MODEL_DIR)
model.bert.save_pretrained(MODEL_DIR / "encoder")

label_config = {
    "sentiment_labels": SENTIMENT_LABELS,
    "impact_labels": IMPACT_LABELS,
    "base_model": BASE_MODEL,
    "max_length": MAX_LENGTH,
}
(MODEL_DIR / "label_config.json").write_text(json.dumps(label_config, indent=2))
print(f"Saved to {MODEL_DIR}")

In [ ]:
# Reload example
loaded = MultiTaskFinBERT(BASE_MODEL)
loaded.load_state_dict(torch.load(MODEL_DIR / "multitask_head.pt", map_location=DEVICE))
loaded = loaded.to(DEVICE)
loaded.eval()

predict_financial_text("Oil prices fell after OPEC hinted at higher output.", model=loaded)

## 7. Next steps

1. **Improve market/sector labels** — add rows to `experiment/data/sample_news_labeled.csv` (or your own CSV) with human annotations.
2. **Train longer** — increase `EPOCHS` to 5–10 once labels are better.
3. **Try LoRA** — fine-tune only adapter layers to save memory (`peft` library).
4. **Integrate with agent** — wrap `predict_financial_text()` as a LangChain tool and pass outputs into your analyser prompt.
5. **Evaluate on your domain** — HK/US headlines behave differently; collect 200+ in-domain labeled samples.

### Suggested labeling guide
- **market_impact**: `high` = macro/policy/market-wide; `medium` = large-cap or index movers; `low` = isolated micro news
- **sector_impact**: `high` = industry-wide shock/regulation; `medium` = major sector player; `low` = company-only or macro-only